In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# STEP 1: Load your existing datasets
df_house = pd.read_csv('houseprice_data.csv')

# Assume you have created these summary files from your other notebooks:
# These should have columns: ['Barangay', 'Flood_Score'] and ['Barangay', 'Avg_AQI']
df_flood = pd.read_csv('barangay_flood_scores.csv') 
df_aqi = pd.read_csv('barangay_aqi_averages.csv')

# STEP 2: Merge the data
# This adds the Flood and AQI data to each house listing based on its location
df = df_house.merge(df_flood, on='Barangay', how='left')
df = df.merge(df_aqi, on='Barangay', how='left')

# Fill any missing scores with the median (for safety)
df['Flood_Score'] = df['Flood_Score'].fillna(df['Flood_Score'].median())
df['Avg_AQI'] = df['Avg_AQI'].fillna(df['Avg_AQI'].median())

# STEP 3: Preprocessing for XGBoost
df['Is Amortized?'] = df['Is Amortized?'].map({'yes': 1, 'no': 0})
df_encoded = pd.get_dummies(df, columns=['Barangay'], drop_first=True)

# Define Features (Now including Flood and AQI)
# We drop Base Price and the calculated sqm columns to avoid 'leaking' the answer
X = df_encoded.drop(['Price', 'Base Price', 'price per lotsqm', 'price per floorsqm'], axis=1)
y = df_encoded['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 4: Train the Enhanced Model
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    reg_lambda=15, 
    importance_type='gain', # This allows us to see how much Flood/AQI matters
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
def analyze_investment_v2(barangay_name):
    # Get current house data + environmental scores
    b_row = df[df['Barangay'] == barangay_name.upper()].iloc[0]
    
    flood = b_row['Flood_Score']
    aqi = b_row['Avg_AQI']
    price = b_row['Price'].mean()
    
    # Adjust Growth Rate based on Risk
    # High Flood Risk (Score 4-5) lowers the 25-year appreciation
    # High Air Quality (Low AQI) increases the appreciation
    base_growth = 0.06
    
    if flood >= 4:
        base_growth -= 0.02  # Deduct 2% growth for flood zones
    if aqi < 0.2: # Assuming 0.2 is 'Very Clean' AOD
        base_growth += 0.01  # Add 1% bonus for clean air
        
    print(f"--- Analysis for {barangay_name} ---")
    print(f"Flood Risk: {flood}/5 | Air Quality Index: {aqi:.3f}")
    print(f"Adjusted Growth Rate: {base_growth*100:.1f}%")
    
    # Project 25 years with the new adjusted rate
    future_25 = price * ((1 + base_growth) ** 25)
    print(f"Estimated Value in 25 Years: PHP {future_25:,.2f}")